# A1c -- hydrophone uptime calendar: inspection

Thin wrapper. All time algebra lives in `boatphone/onc_client.py`; this notebook only calls it
and draws it. Nothing here defines a function.

**What it renders**

1. a year x day-of-year availability heatmap, with ONC deployment boundaries overlaid;
2. the mean-availability-by-UTC-hour profile;
3. a diurnal-null sanity check against a deliberately corrupted (+7 h) calendar.

**Time base: UTC end to end.** Every timestamp below is tz-aware UTC (decision 0002). No local
timezone is referenced anywhere in this notebook or in the library it calls -- an
`America/Vancouver` reading of a UTC stamp is a 7 h error, and plot (2) exists to expose exactly
that.

**This notebook REQUIRES the network and an ONC token.** There is no synthetic fallback: an
invented availability pattern cannot debug a real time base, and a figure that silently degrades
to fiction is exactly how a made-up number gets read as a measurement (invariant 5). If
`ONC_TOKEN` is absent, the client cell raises with a named error rather than drawing something
plausible.

**The span is bounded and stated.** See `SPAN_START_UTC` / `SPAN_END_UTC` below. The full
2020-2026 season calendar -- which takes tens of minutes -- is in
`docs/derived/hydrophone_gaps.md`. Read that file, not this notebook, before
spending Planet quota.

In [ ]:
# Repo root on sys.path, so `import boatphone` works whether the kernel was started
# here or at the repo root. Same pattern as scripts/checks.py. Raises if the
# checkout cannot be located rather than importing something else by that name.
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "boatphone" / "config.py").is_file()), None)
if _root is None:
    raise RuntimeError(
        f"no BoatPhone checkout found at or above {_here}; start the kernel inside the repo"
    )
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"repo root: {_root}")

In [ ]:
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np

from boatphone.config import BIN_SECONDS
from boatphone.credentials import get_onc_client
from boatphone.onc_client import (
    build_uptime_calendar,
    get_deployments,
    mark_available,
    mean_availability_by_utc_hour,
    summarise_gaps,
)

# Named constants for this notebook. Sources in comments (CLAUDE.md invariant 6).

# Span rendered. ONE in-season UTC month, deliberately bounded -- for RUNTIME
# only, so this notebook re-executes in ~a minute. It is no longer a workaround:
# ONC's archive listing caps the rows per response, but list_fft_files now
# follows the response's `next` cursor (and sub-chunks if it cannot) and never
# returns a truncated listing as complete, so a whole-season span is now correct
# too -- 2024-05-01..2024-10-01 is 44,026 files at 99.96 % available in ~70 s
# (measured 2026-08-27; see docs/derived/hydrophone_gaps.md).
SPAN_START_UTC = datetime(2024, 7, 1, tzinfo=timezone.utc)
SPAN_END_UTC = datetime(2024, 8, 1, tzinfo=timezone.utc)

SECONDS_PER_DAY = 24 * 60 * 60
# Gap-reporting threshold. One day is the unit the PlanetScope stream orders in,
# so a shorter gap is not actionable for O1 (docs/plans/Project_Source_of_Truth.txt).
GAP_MIN_SECONDS = SECONDS_PER_DAY
# America/Vancouver summer offset, UTC-7. Used ONLY to build the corrupted null in
# section 5 -- never applied to real data. Written as a number, not a tz lookup:
# boatphone/ is source-scanned to keep zoneinfo out of the analysis path (D4).
PDT_SHIFT_HOURS = 7
DAYS_IN_YEAR_MAX = 366  # leap year; the heatmap grid is fixed-width so years align

## 1. The calendar

`build_uptime_calendar` returns one row per in-season bin: `(start_utc, end_utc, available,
deployment_id)`. It is dense -- an unavailable bin is a row, not a missing row -- so "no data"
and "not measured" stay distinguishable (invariant 9).

`available` means ONC LISTED a `.fft` file overlapping the bin. That is ONC's belief that a file
exists, not proof that it downloads; A4's pull refines it and the pull wins (D3).

The cell below makes network calls and raises if the token or the network is missing.

In [ ]:
client = get_onc_client()  # raises if ONC_TOKEN is absent -- no silent fallback
rows = build_uptime_calendar(client, SPAN_START_UTC, SPAN_END_UTC)
deployments = get_deployments(client)  # ONC METADATA, never inferred from gaps (D6)

bins = [(row[0], row[1]) for row in rows]
available = [row[2] for row in rows]
deployment_ids = [row[3] for row in rows]
source_label = (
    f"ONC listing (MEASURED) {SPAN_START_UTC.date().isoformat()} .. "
    f"{SPAN_END_UTC.date().isoformat()} UTC"
)
print(f"{len(rows)} bin(s) of {BIN_SECONDS} s; {sum(available)} available "
      f"({100.0 * sum(available) / len(rows):.2f}%)")
print(f"{len(deployments)} ONC deployment(s) known for this device")

## 2. Gaps >= 1 day

`summarise_gaps` returns maximal runs of contiguous unavailable bins, `end_utc` EXCLUSIVE. A run
breaks both on an available bin and on a discontinuity in the bin grid itself, so a gap can never
bridge the Sep-to-May season break and report an unmeasured winter as a measured outage.

In [ ]:
gaps = summarise_gaps(bins, available, min_seconds=GAP_MIN_SECONDS)
print(f"{len(gaps)} gap(s) of >= {GAP_MIN_SECONDS} s "
      f"({GAP_MIN_SECONDS / SECONDS_PER_DAY:.0f} day) in {source_label}:")
for gap_start, gap_end, n_bins in gaps:
    days = (gap_end - gap_start).total_seconds() / SECONDS_PER_DAY
    print(f"  {gap_start.isoformat()} -> {gap_end.isoformat()} (exclusive)  "
          f"{n_bins} bin(s), {days:.2f} day(s)")
if not gaps:
    print("  none -- the method ran and found no gap at this threshold "
          "(not the same as the method being broken; invariant 9)")

## 3. Year x day-of-year availability heatmap

One cell per (year, UTC day of year): the fraction of that day's bins marked available. Days with
no bins at all -- everything outside the plotted span, and everything outside the May-September
UTC season -- are left blank rather than drawn as zero, for the same reason the hourly profile
uses `nan`: unmeasured is not empty.

Red marks are ONC deployment boundaries from `get_deployments()` -- deployment METADATA, never
inferred from a gap in the listing (D6). A boundary outside the plotted span is printed rather
than drawn, so a deployment change just off the edge of the figure is still visible to the reader.

In [ ]:
years = sorted({bin_start.year for bin_start, _ in bins})
year_index = {year: i for i, year in enumerate(years)}

day_sum = np.zeros((len(years), DAYS_IN_YEAR_MAX))
day_count = np.zeros((len(years), DAYS_IN_YEAR_MAX))
for (bin_start, _bin_end), flag in zip(bins, available):
    r = year_index[bin_start.year]
    c = bin_start.timetuple().tm_yday - 1
    day_count[r, c] += 1
    day_sum[r, c] += 1.0 if flag else 0.0

with np.errstate(invalid="ignore", divide="ignore"):
    day_fraction = np.where(day_count > 0,
                            day_sum / np.where(day_count > 0, day_count, 1), np.nan)

cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("0.9")  # unmeasured days: grey, visibly different from 0.0 availability

fig, ax = plt.subplots(figsize=(12, 1.6 + 0.6 * len(years)))
mesh = ax.pcolormesh(np.arange(DAYS_IN_YEAR_MAX + 1), np.arange(len(years) + 1),
                     np.ma.masked_invalid(day_fraction), cmap=cmap, vmin=0.0, vmax=1.0)
ax.set_yticks(np.arange(len(years)) + 0.5)
ax.set_yticklabels([str(y) for y in years])
ax.set_xlabel("UTC day of year")
ax.set_ylabel("year")
ax.set_title(f"Hydrophone availability per UTC day -- {source_label}")
fig.colorbar(mesh, ax=ax, label="fraction of day's bins available")

drawn = offscreen = 0
for dep_id, dep_start, dep_end in deployments:
    for edge, kind in ((dep_start, "start"), (dep_end, "end")):
        if edge.year in year_index and SPAN_START_UTC <= edge < SPAN_END_UTC:
            r = year_index[edge.year]
            ax.plot([edge.timetuple().tm_yday - 1 + edge.hour / 24.0] * 2, [r, r + 1],
                    color="red", linewidth=1.5)
            drawn += 1
        else:
            offscreen += 1
        print(f"  deployment {kind}: {edge.isoformat()}  {dep_id}")
print(f"{drawn} deployment boundary/ies drawn (red); {offscreen} outside the plotted span")
plt.show()

## 4. Mean availability by UTC hour -- **this plot is the debugger**

A hydrophone records continuously. It has no preferred hour of the day, so over any span long
enough to average out its start and end, this profile should be **FLAT**.

This figure is not decoration -- it is the check. **If the profile has a dent roughly 7 hours
wide, that is a timezone bug announcing itself, not a diurnal pattern in vessel activity.**
America/Vancouver in summer is UTC-7, so coverage timestamps read as UTC when they were really
local land 7 h late.

**Be precise about the mechanism, because it decides how strong this check is.** A uniform +7 h
shift applied to a *continuously recording* instrument does NOT carve a dent that repeats every
day: the coverage simply slides, and the only bins that change state are at the two ENDS of the
span -- the first 7 h go uncovered and 7 h of coverage falls off the far end. The repeating,
7-hour-wide dent appears only when the span is short enough (or the coverage patchy enough) for
that edge effect to dominate the average, which is exactly the regime the synthetic 7-day check
`check_a1c_15` lives in. On a near-100 %-available month the edge effect is diluted across ~30
days, so **both** the observed profile and the null are nearly flat and the comparison in section
5 is a weak discriminator. Any systematic shape here should still be treated as code until proven
otherwise (invariant 4).

An hour with no bins at all is `nan` and is simply not drawn. Rendering it as 0.0 would draw a
fake outage.

In [ ]:
profile = mean_availability_by_utc_hour(bins, available)
hours = np.arange(24)
values = np.array(profile, dtype=float)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hours, values, marker="o")
ax.set_xticks(hours)
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0.0, 1.05)
ax.set_xlabel("UTC hour of bin start")
ax.set_ylabel("mean availability")
ax.set_title(f"Mean availability by UTC hour -- {source_label}")
ax.grid(alpha=0.3)

measured = values[~np.isnan(values)]
print(f"{len(measured)} of 24 UTC hours have bins; "
      f"{24 - len(measured)} hour(s) are nan (unmeasured, not drawn)")
if len(measured):
    print(f"peak {measured.max():.4f}, trough {measured.min():.4f}, "
          f"peak-to-trough spread {measured.max() - measured.min():.4f}")
plt.show()

## 5. Diurnal null

The observed profile is compared against a deliberately corrupted one. The corruption is applied
to the **coverage timestamps**, shifting them `PDT_SHIFT_HOURS` later and re-running
`mark_available` against the same UTC bin grid -- the same thing `check_a1c_15` does. Shifting the
flag LIST instead would be a different (and wrong) operation: the flag list has no timestamps, and
`season_bins_utc` is discontinuous at 30 Sep -> 1 May, so an index shift across a season boundary
silently moves data across a seven-month jump. Shifting coverage cannot do that -- shifted
coverage that lands outside the season simply covers no bin.

This is a shape comparison, not a hypothesis test. It answers "would this plot notice the bug?"
-- and if the observed spread were as large as the null's, the honest conclusion is to suspect
the code before believing the ocean (invariant 4).

**Read the result with its limitation.** On a span that is ~100 % available, a uniform shift only
changes the bins at the two ends of the span, so the null is itself nearly flat and the two
spreads are both near zero. This check can rule out a gross, span-wide timezone offset; it is
least informative exactly where coverage is best, and it cannot certify a subtler time-base
error. The synthetic 7-day case in `scripts/checks.py` is where the separation is large.

In [ ]:
# Rebuild the coverage this calendar implies -- maximal runs of contiguous available
# bins, half-open [start, end) in UTC -- then shift the COVERAGE (not the flag list)
# by +PDT_SHIFT_HOURS and re-flag against the same bin grid. A run also breaks on a
# discontinuity in the grid itself, so no run bridges the 30 Sep -> 1 May season jump.
coverage_utc = []
run_start_utc = None
prev_end_utc = None
for (bin_start_utc, bin_end_utc), flag in zip(bins, available):
    contiguous = prev_end_utc is not None and bin_start_utc == prev_end_utc
    if flag and (run_start_utc is None or not contiguous):
        if run_start_utc is not None:
            coverage_utc.append((run_start_utc, prev_end_utc))
        run_start_utc = bin_start_utc
    elif not flag and run_start_utc is not None:
        coverage_utc.append((run_start_utc, prev_end_utc))
        run_start_utc = None
    prev_end_utc = bin_end_utc
if run_start_utc is not None:
    coverage_utc.append((run_start_utc, prev_end_utc))

shift = timedelta(hours=PDT_SHIFT_HOURS)
null_coverage_utc = [(cs + shift, ce + shift) for cs, ce in coverage_utc]
null_available = mark_available(bins, null_coverage_utc)
print(f"{len(coverage_utc)} contiguous coverage run(s) reconstructed; "
      f"shifted +{PDT_SHIFT_HOURS} h and re-flagged -> {sum(null_available)} of "
      f"{len(bins)} bins available in the null (observed: {sum(available)})")
null_profile = np.array(mean_availability_by_utc_hour(bins, null_available), dtype=float)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hours, values, marker="o", label="observed")
ax.plot(hours, null_profile, marker="x", linestyle="--",
        label=f"+{PDT_SHIFT_HOURS} h PDT-as-UTC null")
ax.set_xticks(hours)
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0.0, 1.05)
ax.set_xlabel("UTC hour of bin start")
ax.set_ylabel("mean availability")
ax.set_title("Observed profile vs the timezone-error null")
ax.legend()
ax.grid(alpha=0.3)

observed_spread = float(np.nanmax(values) - np.nanmin(values))
null_spread = float(np.nanmax(null_profile) - np.nanmin(null_profile))
print(f"observed peak-to-trough spread: {observed_spread:.4f}")
print(f"+{PDT_SHIFT_HOURS} h null peak-to-trough spread: {null_spread:.4f}")
print("The null is the shape a timezone error would draw. If the observed spread "
      "approaches it, suspect the time base before believing a diurnal signal.")
print("CAVEAT: on a near-100 %-available span a uniform shift only changes the bins at the "
      "two ENDS of the span, so both spreads are near zero and this is a WEAK discriminator. "
      "It rules out a gross span-wide offset; it does not certify the time base.")
plt.show()

## What this notebook does and does not claim

* Availability here is ONC's **listing**, not a completed download. An absent listing entry is not
  a verified absent file; A4's pull may revise it (D3), and the pull wins.
* The figures above cover ONE bounded UTC month, named in `SPAN_START_UTC` / `SPAN_END_UTC`.
  They are not the full calendar and must not be read as one.
* The gap spans and available fractions that ship to the PlanetScope stream are in
  `docs/derived/hydrophone_gaps.md`, together with the span they were measured over.
* Nothing here defines a function: `summarise_gaps` and `mean_availability_by_utc_hour` live in
  `boatphone/onc_client.py` so A4 can reuse them and a check can reach them.